# Porównanie Sieci MLP: Float32 vs Fixed-Point (PTQ i QAT)
W tym projekcie badamy wpływ kwantyzacji na rozmiar i dokładność prostej sieci neuronowej trenowanej na zbiorze MNIST.

In [ ]:
import os
import copy
import time # NOWE: do mierzenia czasu
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tabulate import tabulate

BATCH_SIZE = 64
FP32_EPOCHS = 5
QAT_EPOCHS = 2
DEVICE = torch.device('cpu')

supported_engines = torch.backends.quantized.supported_engines
if 'x86' in supported_engines:
    Q_ENGINE = 'x86'
elif 'fbgemm' in supported_engines:
    Q_ENGINE = 'fbgemm'
elif 'qnnpack' in supported_engines:
    Q_ENGINE = 'qnnpack'
else:
    Q_ENGINE = supported_engines[0]

torch.backends.quantized.engine = Q_ENGINE
print(f"Wybrano silnik: {Q_ENGINE}")

Dostępne silniki kwantyzacji w Twoim systemie: ['onednn']
Wybrano silnik: onednn


In [13]:
class QuantizedMLP(nn.Module):
    def __init__(self):
        super(QuantizedMLP, self).__init__()
        # QuantStub: wejście -> fixed point
        self.quant = torch.ao.quantization.QuantStub()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)
        # DeQuantStub: fixed point -> wyjście (float)
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.quant(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.dequant(x)
        return x

# Ładowanie danych
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Dane załadowane! Ilość paczek treningowych: {len(train_loader)}")

Dane załadowane! Ilość paczek treningowych: 938


In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs=1):
    model.train()
    for epoch in range(epochs):
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        print(f"Zakończono epokę {epoch+1}/{epochs}")

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(test_loader.dataset)

def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    size = os.path.getsize("temp.p") / 1e3
    os.remove("temp.p")
    return size

def measure_inference_time(model, test_loader):
    model.eval()
    with torch.no_grad():
        for _ in range(5):
            dummy_data = next(iter(test_loader))[0]
            _ = model(dummy_data)
            
    start_time = time.perf_counter()
    with torch.no_grad():
        for data, _ in test_loader:
            _ = model(data)
    end_time = time.perf_counter()
    
    return (end_time - start_time) * 1000

### Eksperyment 1: Sieć bazowa Float32 (FP32)
Trenujemy standardową sieć bez żadnych modyfikacji.

In [ ]:
criterion = nn.CrossEntropyLoss()

print("--- Trenowanie modelu bazowego (Float 32) ---")
model_fp32 = QuantizedMLP().to(DEVICE)
optimizer = optim.Adam(model_fp32.parameters(), lr=0.001)

train_model(model_fp32, train_loader, criterion, optimizer, epochs=FP32_EPOCHS)

acc_fp32 = evaluate_model(model_fp32, test_loader)
size_fp32 = print_size_of_model(model_fp32)
print(f"FP32 Dokładność: {acc_fp32:.2f}%, Rozmiar: {size_fp32:.2f} KB")

--- Trenowanie modelu bazowego (Float 32) ---
Zakończono epokę 1/10
Zakończono epokę 2/10
Zakończono epokę 3/10
Zakończono epokę 4/10
Zakończono epokę 5/10
Zakończono epokę 6/10
Zakończono epokę 7/10
Zakończono epokę 8/10
Zakończono epokę 9/10
Zakończono epokę 10/10
FP32 Dokładność: 97.55%, Rozmiar: 440.24 KB


### Eksperyment 2: Post-Training Quantization (PTQ)
Kopiujemy wytrenowany model FP32, kalibrujemy go na danych testowych i konwertujemy jego wagi na postać stałoprzecinkową (INT8).

In [16]:
print("--- Post-Training Quantization (PTQ) ---")
model_ptq = copy.deepcopy(model_fp32)
model_ptq.eval()

# Konfiguracja i przygotowanie
model_ptq.qconfig = torch.ao.quantization.get_default_qconfig(Q_ENGINE)
torch.ao.quantization.prepare(model_ptq, inplace=True)

# Kalibracja
print("Kalibracja modelu PTQ...")
evaluate_model(model_ptq, test_loader)

# Konwersja na fixed-point
torch.ao.quantization.convert(model_ptq, inplace=True)

acc_ptq = evaluate_model(model_ptq, test_loader)
size_ptq = print_size_of_model(model_ptq)
print(f"PTQ Dokładność: {acc_ptq:.2f}%, Rozmiar: {size_ptq:.2f} KB")

--- Post-Training Quantization (PTQ) ---
Kalibracja modelu PTQ...


C:\Users\Kamil\AppData\Local\Temp\ipykernel_53820\815221552.py:6: UserWarning: Default qconfig of oneDNN backend with reduce_range of false may have accuracy issues on CPU without Vector Neural Network Instruction support.
  model_ptq.qconfig = torch.ao.quantization.get_default_qconfig(Q_ENGINE)
C:\Users\Kamil\AppData\Local\Temp\ipykernel_53820\815221552.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorc

PTQ Dokładność: 97.53%, Rozmiar: 119.45 KB


### Eksperyment 3: Quantization-Aware Training (QAT)
Kopiujemy bazowy model, ale poddajemy go dalszemu trenowaniu symulując obcięcie precyzji w trakcie wstecznej propagacji błędu.

In [ ]:
print("--- Quantization-Aware Training (QAT) ---")
model_qat = copy.deepcopy(model_fp32)
model_qat.train()

# Konfiguracja
model_qat.qconfig = torch.ao.quantization.get_default_qat_qconfig(Q_ENGINE)
torch.ao.quantization.prepare_qat(model_qat, inplace=True)

# Dotrenowywanie (Fake Quantization) z mniejszym learning rate
print("Dotrenowywanie modelu QAT...")
optimizer_qat = optim.Adam(model_qat.parameters(), lr=0.0001)
train_model(model_qat, train_loader, criterion, optimizer_qat, epochs=QAT_EPOCHS)

# Konwersja na gotowy model
model_qat.eval()
torch.ao.quantization.convert(model_qat, inplace=True)

acc_qat = evaluate_model(model_qat, test_loader)
size_qat = print_size_of_model(model_qat)
print(f"QAT Dokładność: {acc_qat:.2f}%, Rozmiar: {size_qat:.2f} KB")

--- Quantization-Aware Training (QAT) ---
Dotrenowywanie modelu QAT...


C:\Users\Kamil\AppData\Local\Temp\ipykernel_53820\2590176606.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.prepare_qat(model_qat, inplace=True)


Zakończono epokę 1/1


C:\Users\Kamil\AppData\Local\Temp\ipykernel_53820\2590176606.py:16: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.ao.quantization.convert(model_qat, inplace=True)


QAT Dokładność: 98.15%, Rozmiar: 119.45 KB


### Podsumowanie i Wnioski
Zestawienie spadku dokładności w stosunku do zysku na rozmiarze modelu w pamięci.

In [18]:
results = [
    ["Float32 (Baza)", f"{acc_fp32:.2f}%", f"{size_fp32:.2f} KB"],
    ["PTQ (Post-Training)", f"{acc_ptq:.2f}%", f"{size_ptq:.2f} KB"],
    ["QAT (Aware Training)", f"{acc_qat:.2f}%", f"{size_qat:.2f} KB"]
]
print(tabulate(results, headers=["Model Typ", "Dokładność", "Rozmiar Pamięci"], tablefmt="grid"))

+----------------------+--------------+-------------------+
| Model Typ            | Dokładność   | Rozmiar Pamięci   |
+======================+==============+===================+
| Float32 (Baza)       | 97.55%       | 440.24 KB         |
+----------------------+--------------+-------------------+
| PTQ (Post-Training)  | 97.53%       | 119.45 KB         |
+----------------------+--------------+-------------------+
| QAT (Aware Training) | 98.15%       | 119.45 KB         |
+----------------------+--------------+-------------------+
